In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.24 Green's Functions: The Propagator at Temperature

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.24",
    title="Green's Functions: The Propagator at Temperature",
    blurb="Experiments rarely ask a Hamiltonian for its eigenvalues; they add an electron "
    "or remove one and watch what it costs. The Green's function is that question "
    "made mathematics — a propagator living on the thermal circle, antiperiodic, "
    "with the anticommutator hiding in its jump. We make its definition fully "
    "numerical, put Matsubara sums to work at last — including one that converges "
    "beautifully to the wrong answer by exactly one half — watch the Mott gap open "
    "in a spectrum where last notebook it hid in a specific heat, and quantify, to "
    "three digits, exactly why extracting spectra from imaginary-time data is a "
    "famously treacherous inverse problem.",
    difficulty="advanced",
    estimate="205–245 min",
    optional=True,
)

## Notebook overview

The Coda's second notebook teaches the sentence its alphabet exists for. [§7.23](second-quantization.ipynb) built
operators that add and remove quanta; the **Green's function** is what those operators
say when a system at temperature is actually asked: add one particle (or remove one),
wait an imaginary time $\tau$, take it back out, and record the amplitude. Every spectrum
in this course so far was obtained by asking a Hamiltonian for its eigenvalues — but
experiments rarely ask that way. A photoemission lamp *removes* an electron; a tunneling
tip *adds* one; what either measures is the energy cost and amplitude of changing the
particle number by one, in a system already thermal. The propagator
$G_{ij}(\tau) = -\langle T_\tau\, c_i(\tau) c_j^\dagger(0)\rangle$ is that measurement
made mathematics, and this notebook makes every one of its properties numerical.

Two long-running threads finally do a day's work. **Matsubara frequencies** — produced by
the contour ingenuity of [§7.2](complex-analysis-applications.ipynb), explained by the thermal circle of [§7.20](imaginary-time-quantum-classical.ipynb) — are revealed as simply the
addresses where $G$ lives, and Matsubara *sums* are performed three ways, with a verified
surprise as the teacher: the textbook frequency sum for the occupation, truncated
symmetrically without its convergence factor, converges cleanly to the *wrong answer by
exactly one half* — and the missing half is precisely the equal-time jump that the
anticommutator plants in $G$'s discontinuity. A convergent-looking number is not thereby
a correct one; convergence tests precision, sum rules test truth. And the **gap-from-decay**
of [§7.21](path-integral-monte-carlo.ipynb) is placed where it belongs: as the one benign case of an inversion whose
general ill-posedness this notebook elevates from slogan to verified law, with two
grossly different spectra producing Matsubara data that agree to parts in $10^6$ — and a
moment-suppression formula that predicts the difference to three digits at every
frequency but the lowest.

The centerpiece keeps the promise of [§7.23](second-quantization.ipynb): the **Hubbard dimer's Green's function** at the
particle–hole point. The free limit certifies the machinery (two poles at $\pm t$, the
molecular-orbital levels re-read as addition and removal energies), a four-part sum-rule
battery is passed before anything about interactions is believed, and then the verified
teaching surprise: at $U = 8t$ and $\beta = 4$ the spectrum shows *eight* poles, not the
textbook four, because the thermally populated triplet adds satellite lines. The spectral
function is a thermal object; textbooks draw its $T = 0$ skeleton. Cooling to
$\beta = 20$ collapses the satellites and the four-pole skeleton stands: the lower and
upper **Hubbard bands**, split by $\sim U$ — the Mott gap that [§7.23](second-quantization.ipynb) read from a specific
heat, now visible in a spectrum. Same physics, new instrument, and the correspondence is
tabled.

> **Conventions (this notebook).** Fermionic conventions fixed once: $\zeta = -1$;
> $G_{ij}(\tau) = -\langle T_\tau c_i(\tau) c_j^\dagger(0)\rangle$ with
> $c(\tau) = e^{\tau K} c\, e^{-\tau K}$, $K = H - \mu N$, and $T_\tau$ ordering larger
> $\tau$ to the left with a sign per exchange; for $0 < \tau < \beta$,
> $G(\tau) = -\mathrm{Tr}[e^{-\beta K} e^{\tau K} c\, e^{-\tau K} c^\dagger]/Z$. Every
> spectral evaluation ground-shifts its exponentials ($e^{-\beta(K_m - K_0)}$ and
> $e^{\tau(K_m - K_n)}$ arranged so no argument is large and positive — the discipline of [§7.4](thermal-density-matrix.ipynb);
> $\beta = 20$ underflows without it). Matsubara frequencies are fermionic,
> $\omega_n = (2n{+}1)\pi/\beta$. The Lehmann evaluation is one vectorized
> weight-matrix-over-pole-matrix expression (the quadruple loop is named and forbidden).
> Spectral poles are consolidated by `numpy.unique` on `numpy.round(poles, 8)` with the
> sum rule re-checked after consolidation. Fourier transforms on $[0, \beta]$ use
> `numpy.trapezoid` with endpoints included. The operator machinery of [§7.23](second-quantization.ipynb) (`kron_chain`,
> `jw_ops`, the dimer builder, sector logic) is reused verbatim, restated here with
> provenance so the notebook runs standalone. The dimer sits at $\mu = U/2$ throughout.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the one-mode propagator against its closed form and its jump; the
> Matsubara routes against $n_F$ (and the naive route against its wrong constant); the
> Lehmann code against the free chain's mode sum; the dimer against its sum-rule battery,
> its pole count, and its Hubbard-band closed form; the quadrature against the pole
> representation at second order; the ill-posedness against its moment law. A ✓ is
> strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** Analytic continuation in practice (MaxEnt, stochastic methods, Nevanlinna:
> Jarrell & Gubernatis, *Phys. Rep.* **269**, 133 (1996) is the canon), self-energies
> and Dyson's equation, and ARPES/STM as laboratories for $A(\omega)$ are outward
> horizons; linear response is the Coda's last notebook. See Fetter & Walecka Ch. 7–9;
> Mahan, *Many-Particle Physics*; Bruus & Flensberg for the pedagogical route.
> Cross-reference [§7.2](complex-analysis-applications.ipynb) (the frequencies, now put to work), [§7.20](imaginary-time-quantum-classical.ipynb) (the thermal circle and
> KMS, here in fermionic edition), [§7.21](path-integral-monte-carlo.ipynb) (the benign case, placed), [§7.23](second-quantization.ipynb) (the machinery
> reused; the dimer promise kept; the two-scale table), [§7.7](bose-einstein-fermi-dirac.ipynb) ($n_F$, everywhere), and
> forward to [§7.25](linear-response-kubo.ipynb) (response).

## Theory in brief

### The question experiments actually ask

A Hamiltonian's eigenvalues answer "what energies can this system have?" — but a
photoemission experiment removes an electron and a tunneling tip injects one, so the
question actually posed is "what does changing the particle number by one cost, and with
what amplitude, at temperature?" The object that answers is the imaginary-time
single-particle Green's function,

```{math}
:label: eq-gf-definition
G_{ij}(\tau) \;=\; -\big\langle T_\tau\, c_i(\tau)\, c_j^\dagger(0)\big\rangle
\;=\; -\frac{1}{Z}\,\mathrm{Tr}\Big[e^{-\beta K}\, e^{\tau K} c_i\, e^{-\tau K} c_j^\dagger\Big]
\quad (0 < \tau < \beta),
```

with $K = H - \mu N$ and $T_\tau$ the imaginary-time ordering (a fermionic sign per
exchange). Its pole structure — the spectral function $A(\omega)$ below — is what ARPES
and STM measure: this notebook's object is an observable, not a formal device.

### The free mode: everything checkable

One fermionic mode at energy $\varepsilon$ is the hydrogen atom of Green's functions,
because the trace has two states and every claim can be checked to the last digit.
Evaluating {eq}`eq-gf-definition` on the two-state Fock space gives

```{math}
:label: eq-free-mode
G(\tau) = -e^{-\varepsilon\tau}\big(1 - n_F(\varepsilon)\big),
\qquad
G(0^+) + G(\beta^-) = -1,
\qquad
G(\tau + \beta) = -G(\tau),
```

and each of the three statements carries a lesson. The first is the propagator itself
(verified below against ED at $10^{-16}$). The second — the **jump rule** — is the
anticommutator $\{c, c^\dagger\} = 1$ living in the $\tau$-discontinuity: a sum rule
before any integral, and the diagnostic key for a failure mode two sections down. The
third is the KMS circle of [§7.20](imaginary-time-quantum-classical.ipynb) in fermionic edition, the minus sign falling out of the
anticommutation inside the cyclic trace in three lines. Antiperiodic functions have
odd-harmonic Fourier content only, so the fermionic frequencies of [§7.2](complex-analysis-applications.ipynb)
$\omega_n = (2n{+}1)\pi/\beta$ — twice introduced, never yet worked — are simply the
addresses where $G$ lives: $G(i\omega_n) = 1/(i\omega_n - \varepsilon)$, verified by
direct quadrature.

### Matsubara sums for a living

Every finite-temperature perturbation theory sooner or later evaluates a frequency sum,
and the simplest one already contains the trade's whole craft. The occupation is the
textbook identity $n_F(\varepsilon) = T\sum_n e^{i\omega_n 0^+}/(i\omega_n - \varepsilon)$
(Mahan Ch. 3 derives it by contour), and the innocuous-looking $0^+$ is the entire
content:

```{math}
:label: eq-matsubara-sums
T\!\!\sum_{n=-N}^{N-1}\frac{1}{i\omega_n - \varepsilon}
\;\xrightarrow{\ N\to\infty\ }\; n_F(\varepsilon) - \tfrac12,
\qquad\text{vs}\qquad
\tfrac12 + T\sum_{n\ge0}\frac{-2\varepsilon}{\omega_n^2 + \varepsilon^2}
\;\to\; n_F(\varepsilon).
```

The naive symmetric truncation on the left converges *cleanly to the wrong answer*,
off by exactly the jump rule's $\tfrac12$: the dropped convergence factor is what decides
whether the equal-time limit is taken from $0^-$ or $0^+$, and half the discontinuity is
the price of forgetting it. The $\pm n$ paired form on the right converges at a measured
$1/N$ rate to the truth. Three routes (naive, paired, contour closed form), one number,
and a standing kit: pair frequencies, know the tail, respect the $0^+$.

### Lehmann made numerical

Where do $G$'s poles come from? Insert energy eigenstates into {eq}`eq-gf-definition`
in both orderings and Fourier transform; the result is the Lehmann representation,

```{math}
:label: eq-lehmann
G_{ij}(i\omega_n) \;=\; \frac{1}{Z}\sum_{m,n}
\frac{\big(e^{-\beta K_m} + e^{-\beta K_n}\big)\,
\langle m|c_i|n\rangle\langle n|c_j^\dagger|m\rangle}{i\omega_n - (K_n - K_m)},
```

a sum of simple poles at the *differences* of many-body energies, weighted by matrix
elements and thermal factors. Numerically this is one vectorized expression — a weight
matrix over a pole matrix — and the quadruple-loop temptation is forbidden on cost
grounds alone. The pole–weight list *is* the spectral function $A(\omega)$, and the
machinery is certified on the free four-site chain against the mode sum
$\sum_k |U_{0k}|^2/(i\omega_n - \varepsilon_k)$: the interacting-capable code passing
the exactly solvable exam, the course's standard rite.

### The dimer's Green's function (centerpiece)

[§7.23](second-quantization.ipynb) promised its interacting flagship a spectrum, and the grand-canonical stage is set
by one choice: at $\mu = U/2$ the dimer is particle–hole symmetric (two lines below),
every spectral function is even, and half filling is exact at every temperature. The
free limit gives exactly two poles at $\pm t$ — the molecular-orbital levels, re-read as
addition and removal energies — and a four-part **sum-rule battery** must pass before
interactions are believed: total weight one (the anticommutator again), first moment
zero, evenness, and $n = \tfrac12$ read off the propagator. Then the interacting
spectrum, where the honest headline is that

```{math}
:label: eq-dimer-gf
A(\omega)\ \text{is a thermal object:}
\qquad
\omega_{\text{bands}} = \pm\frac{\sqrt{U^2 + 16t^2} \mp 2t}{2},
\qquad
w_{\pm} = \frac14\Big(1 \pm \frac{4t}{\sqrt{U^2 + 16t^2}}\Big),
```

with the closed forms holding for the $T\to0$ skeleton — and at $\beta = 4$ the computed
spectrum showing *eight* poles instead of four, because the triplet a mere
$J = 4t^2/U$ above the singlet is thermally populated and its transitions add satellite
lines. Textbooks draw the skeleton; the object itself wears its temperature. Cooling to
$\beta = 20$ kills the satellites and the skeleton stands: lower and upper Hubbard
bands, split by $\sim U$ — the Mott gap of the two-peak specific heat of [§7.23](second-quantization.ipynb), now in a
spectrum, with the correspondence tabled ($U \leftrightarrow$ band splitting
$\leftrightarrow$ the charge peak; $J \leftrightarrow$ satellite activation
$\leftrightarrow$ the spin peak).

### The Fourier rendezvous

The same $G(i\omega_n)$ must arrive by residues and by quadrature, and checking that it
does is the notebook's internal-consistency seal:

```{math}
:label: eq-ft-rendezvous
G(i\omega_n) \;=\; \int_0^\beta d\tau\, G(\tau)\, e^{i\omega_n\tau}
\quad\text{(trapezoid)}
\qquad\text{vs}\qquad
\text{Lehmann poles},
```

converging at second order in the grid spacing (verified: $2\times10^{-5} \to
7.7\times10^{-8}$ across $401 \to 6401$ points). Trapezoid suffices on the *closed*
interval $[0, \beta]$ because $G$ itself is smooth there — only its derivative jumps at
the endpoints, which second-order quadrature tolerates.

### The ill-posed inverse, as a law

Finally, the honest accounting of what this object refuses to do. Inverting Matsubara
data back to a spectrum is famously treacherous, and the notebook elevates the slogan to
a verified law. Expand the kernel in $1/\omega_n$:

```{math}
:label: eq-ill-posed
G(i\omega_n) = \int\!d\omega\,\frac{A(\omega)}{i\omega_n - \omega}
= \frac{M_0}{i\omega_n} + \frac{M_1}{(i\omega_n)^2} + \frac{M_2}{(i\omega_n)^3} + \cdots
\;\;\Longrightarrow\;\;
|\Delta G| \approx \frac{\Delta M_2}{\omega_n^3},
```

so two spectra with matched low moments differ in their data only through higher moments,
suppressed by powers of $\omega_n$. Built below: one spectrum with weight split at
$\pm0.2$, one with a single peak at $0$ — moments matched through $M_1$ — whose Matsubara
data differ by $10^{-2}$ at $n = 0$ falling to $8\times10^{-6}$ at $n = 5$, with the
moment law matching the measured differences to three digits at every frequency but the lowest. The
kernel transmits moments and exponentially little else; inversion is ill-posed by
arithmetic, not bad luck. The gap-from-decay of [§7.21](path-integral-monte-carlo.ipynb) is then placed precisely: a single
dominant pole is the one benign case (one exponential, one rate — the late-$\tau$ limit
performs the regularization physically), and MaxEnt and Nevanlinna are the field's honest
responses (Jarrell & Gubernatis 1996).

### The gateway reading

The alphabet made sentences, and they turned out to be the sentences experiments speak.
One notebook remains in the gateway: perturb the system weakly, and the response was
already written in equilibrium's correlation functions ([§7.25](linear-response-kubo.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Fermionic conventions fixed once: zeta = -1; G(tau) = -<T c(tau) c+(0)>;
# K = H - mu N; omega_n = (2n+1) pi / beta. Every spectral evaluation is
# ground-shifted (the discipline of §7.4): beta = 20 underflows without it.

# --- the operator machinery of §7.23, reused verbatim (restated with provenance
# --- so this notebook runs standalone; the constructions and conventions
# --- are those of §7.23, including mode 0 = leftmost kron factor).
C1 = np.array([[0.0, 1.0], [0.0, 0.0]])
Z1 = np.array([[1.0, 0.0], [0.0, -1.0]])
I1 = np.eye(2)


def kron_chain(ops):
    """Kronecker product of a list of matrices, left to right (§7.23)."""
    out = np.array([[1.0]])
    for o in ops:
        out = np.kron(out, o)
    return out


def jw_ops(M):
    """Jordan-Wigner annihilators c_i = Z x...x Z x c x I x...x I (§7.23).

    §7.23 established this as the definition of lattice fermions and issued
    the anticommutator table as the standing unit test; the gate is re-run
    below once, on import, before anything else is trusted.

    Parameters
    ----------
    M : int
        Number of modes.

    Returns
    -------
    list of numpy.ndarray
        The M annihilators.
    """
    return [kron_chain([Z1] * i + [C1] + [I1] * (M - i - 1)) for i in range(M)]


CS = jw_ops(4)
N_OP = sum(c.T @ c for c in CS)
N_DIAG = np.round(np.diag(N_OP)).astype(int)  # round, never cast (§7.23)
_gate = max(
    max(
        np.abs(ci @ cj.T + cj.T @ ci - np.eye(16) * (i == j)).max(),
        np.abs(ci @ cj + cj @ ci).max(),
    )
    for i, ci in enumerate(CS)
    for j, cj in enumerate(CS)
)
print(f"the anticommutator gate of §7.23, re-run on import: {_gate:.1e}")


def hubbard_dimer(t, U):
    """The Hubbard dimer on modes [1up, 1dn, 2up, 2dn] (the builder of §7.23).

    H = -t sum_sigma (c+_{1s} c_{2s} + h.c.) + U (n1up n1dn + n2up n2dn).

    Parameters
    ----------
    t, U : float
        Hopping and on-site repulsion.

    Returns
    -------
    numpy.ndarray
        The 16-state Fock Hamiltonian.
    """
    H = np.zeros((16, 16))
    for i, j in ((0, 2), (1, 3)):
        H += -t * (CS[i].T @ CS[j] + CS[j].T @ CS[i])
    n = [c.T @ c for c in CS]
    H += U * (n[0] @ n[1] + n[2] @ n[3])
    return H


# --- this notebook's own machinery ---
def G_tau(taus, evals, c_mn, beta):
    """G(tau) = -<T c(tau) c+(0)> for 0 < tau < beta, by spectral decomposition.

    Ground-shifted throughout: with K_m' = K_m - K_0, the summand is
    exp(-beta K_m') * exp(tau (K_m' - K_n')) — every exponent bounded above
    by 0 on 0 < tau < beta, so beta = 20 cannot underflow to garbage
    (the discipline of §7.4; the naive form dies there).

    Parameters
    ----------
    taus : numpy.ndarray
        Times in (0, beta).
    evals : numpy.ndarray
        Eigenvalues of K.
    c_mn : numpy.ndarray
        <m|c|n> in the K eigenbasis.
    beta : float
        Inverse temperature.

    Returns
    -------
    numpy.ndarray
        G(tau) on the grid.
    """
    k = evals - evals[0]
    w = np.exp(-beta * k)
    Z = w.sum()
    out = np.empty(len(taus))
    for idx, tau in enumerate(np.atleast_1d(taus)):
        out[idx] = (
            -np.sum(w[:, None] * np.exp(tau * (k[:, None] - k[None, :])) * c_mn**2) / Z
        )
    return out


def lehmann(z, evals, c_mn, beta):
    """G(z) as the vectorized weight-matrix-over-pole-matrix expression.

    W_mn = (e^{-beta K_m} + e^{-beta K_n}) |<m|c|n>|^2 / Z (ground-shifted),
    P_mn = K_n - K_m; G(z) = sum W / (z - P). One outer expression — the
    quadruple loop over (m, n, i, j) is forbidden: it recomputes the same
    matrix elements D^2 times and turns a millisecond into minutes.

    Parameters
    ----------
    z : complex
        Frequency (typically i omega_n).
    evals : numpy.ndarray
        Eigenvalues of K.
    c_mn : numpy.ndarray
        <m|c|n> in the K eigenbasis.
    beta : float
        Inverse temperature.

    Returns
    -------
    complex
        G(z).
    """
    k = evals - evals[0]
    w = np.exp(-beta * k)
    Z = w.sum()
    W = (w[:, None] + w[None, :]) * c_mn**2 / Z
    P = k[None, :] - k[:, None]
    return complex(np.sum(W / (z - P)))


def spectral_poles(evals, c_mn, beta, wtol=1e-10, rnd=8):
    """A(omega) as consolidated poles and weights from the Lehmann sum.

    Raw pole positions K_n - K_m are consolidated by numpy.unique on
    numpy.round(poles, rnd): too tight a rounding splits one physical pole
    into numerical twins, too loose merges distinct physics — rnd = 8 with
    the sum rule re-checked after consolidation is the stated policy.
    Weights below wtol are dropped AFTER consolidation (satellite poles are
    physics, not noise: the threshold must sit far below any thermal
    weight one intends to keep).

    Parameters
    ----------
    evals : numpy.ndarray
        Eigenvalues of K.
    c_mn : numpy.ndarray
        <m|c|n> in the K eigenbasis.
    beta : float
        Inverse temperature.
    wtol : float, optional
        Weight floor (default 1e-10).
    rnd : int, optional
        Consolidation rounding digits (default 8).

    Returns
    -------
    poles, weights : numpy.ndarray
        The spectral function as sticks, sorted by pole position.
    """
    k = evals - evals[0]
    w = np.exp(-beta * k)
    Z = w.sum()
    W = (w[:, None] + w[None, :]) * c_mn**2 / Z
    P = k[None, :] - k[:, None]
    flat_p, flat_w = P.ravel(), W.ravel()
    uniq, inv = np.unique(np.round(flat_p, rnd), return_inverse=True)
    wts = np.zeros_like(uniq)
    np.add.at(wts, inv, flat_w)
    keep = wts > wtol
    return uniq[keep], wts[keep]


def matsubara_nF_naive(eps, beta, N):
    """The WRONG route: symmetric truncation without the 0+ factor.

    Kept because the failure is the lesson: this converges cleanly to
    n_F - 1/2 (Exercise 2 demonstrates and diagnoses).
    """
    ns = np.arange(-N, N)
    wn = (2 * ns + 1) * np.pi / beta
    return float(np.sum((1.0 / (1j * wn - eps)).real) / beta)


def matsubara_nF_paired(eps, beta, N):
    """n_F by the +-n paired Matsubara sum: 1/2 + T sum_{n>=0} -2eps/(wn^2+eps^2).

    Pairing n with -n-1 makes each term real and O(1/wn^2), so the truncated
    sum converges at 1/N with a tail T*2eps/(pi^2) * (beta/(2N)) — the rate
    Exercise 2 measures.

    Parameters
    ----------
    eps, beta : float
        Mode energy and inverse temperature.
    N : int
        Number of paired terms.

    Returns
    -------
    float
        The truncated paired sum.
    """
    wn = (2 * np.arange(N) + 1) * np.pi / beta
    return float(0.5 + np.sum(-2 * eps / (wn**2 + eps**2)) / beta)


def ft_G(tau_grid, G_grid, wn):
    """G(i wn) by numpy.trapezoid of G(tau) e^{i wn tau} on [0, beta].

    Endpoints included: G is smooth on the closed interval (only its
    derivative jumps at 0 and beta), so trapezoid keeps its second order.

    Parameters
    ----------
    tau_grid, G_grid : numpy.ndarray
        The grid and G(tau) on it.
    wn : float
        Fermionic Matsubara frequency.

    Returns
    -------
    complex
        The quadrature G(i wn).
    """
    return complex(np.trapezoid(G_grid * np.exp(1j * wn * tau_grid), tau_grid))


def sum_rules(poles, weights, beta):
    """The battery: total weight, first moment, PH asymmetry, and n = G(0-).

    n is read off the spectral representation as sum_i w_i n_F(pole_i):
    the occupation off the propagator, the practical rule of Exercise 4.

    Parameters
    ----------
    poles, weights : numpy.ndarray
        The stick spectrum.
    beta : float
        Inverse temperature.

    Returns
    -------
    tuple of float
        (sum A, first moment, max PH asymmetry, n).
    """
    sumA = float(weights.sum())
    M1 = float(np.sum(poles * weights))
    ph = float(np.abs(np.sort(poles) + np.sort(-poles)[::-1]).max())
    n_occ = float(np.sum(weights / (np.exp(beta * poles) + 1.0)))
    return sumA, M1, ph, n_occ

## Exercise 1 — The propagator at temperature, certified on one mode

Definitions with every claim checkable. Cite {eq}`eq-gf-definition`, {eq}`eq-free-mode`.

1. Fix the conventions (ordering, trace, sign — the Setup block) and compute $G(\tau)$
   for a single mode at $\varepsilon = 0.8$, $\beta = 2$ by two-state spectral
   decomposition (`G_tau`); verify against $-e^{-\varepsilon\tau}(1 - n_F)$.
2. Verify the jump rule $G(0^+) + G(\beta^-) = -1$: derive it from
   $\{c, c^\dagger\} = 1$, then check numerically.
3. Derive antiperiodicity in three lines (the fermionic minus inside the cyclic trace)
   and verify it; verify the odd-only Fourier content of the $2\beta$-extended function
   (`numpy.trapezoid` coefficients at $k = 1$ and $k = 2$) — the circle of [§7.20](imaginary-time-quantum-classical.ipynb), fermionic
   edition.
4. Compute $G(i\omega_n)$ by direct quadrature (`ft_G`) and verify
   $1/(i\omega_n - \varepsilon)$ at $n = 0$ and $3$; state where $G$ lives.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    [jump, G_iw0.real, G_iw0.imag],
    [-1.0, ex_iw0.real, ex_iw0.imag],
    "the propagator certified on one mode: the jump and the address",
    rtol=1e-6,
)
validate.check(
    free_dev < 1e-14 and anti_dev < 1e-8 and c_even < 1e-3 * c_odd,
    "closed form met, antiperiodic on the circle, odd harmonics only",
    f"G dev {free_dev:.1e}; KMS dev {anti_dev:.1e}; even/odd = {c_even/c_odd:.1e}",
)

## Exercise 2 — A sum that converges to the wrong answer

Matsubara sums for a living: the $0^+$ trap demonstrated, diagnosed, repaired. Cite
{eq}`eq-matsubara-sums`.

1. Evaluate $n_F = T\sum_n e^{i\omega_n 0^+}/(i\omega_n - \varepsilon)$ by naive
   symmetric truncation (`matsubara_nF_naive`, the $0^+$ dropped) at
   $N = 10^2, 10^3, 10^4$ and demonstrate the failure: clean convergence to
   $n_F - \tfrac12$.
2. Diagnose via Exercise 1's jump rule: the $0^+$ factor decides which side of the
   equal-time discontinuity the sum lands on, and dropping it splits the difference —
   derive the connection.
3. Repair by $\pm n$ pairing (`matsubara_nF_paired`; derive the paired form) and measure
   the $1/N$ convergence; state the tail estimate.
4. Close the three-route rendezvous with the contour closed form and issue the kit:
   pair frequencies, know the tail, respect the $0^+$ — and the meta-rule that
   convergence tests precision while sum rules test truth.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [naive_vals[10000], paired_final],
    [NF1 - 0.5, NF1],
    "the 0+ trap: convergent, wrong by exactly one half, diagnosed, repaired",
    rtol=1e-3,
)
validate.check(
    50 < rate < 200,
    "and the paired route converges at the predicted 1/N",
    f"error fell {rate:.0f}x across 100x more terms",
)

```{admonition} With your assistant
:class: tip
Ask your assistant for a Matsubara-sum snippet for $\langle n \rangle$ — the
sum Exercise 2 just performed with such care — and run what it produces. Then
aim this exercise's own lesson at it: does the generated sum carry the
$e^{i\omega_n 0^+}$ convergence factor, or has it quietly summed to the wrong
answer with perfect numerical confidence? Check it against the exact Fermi
function before believing a digit. Convergent and correct are different
words — for your code and its code alike. The check is yours.
```

## Exercise 3 — Lehmann, made numerical and certified

The definition as one vectorized expression, passed through the free-chain exam. Cite
{eq}`eq-lehmann`.

1. Derive the Lehmann representation: insert eigenstates of $K$ into both orderings of
   {eq}`eq-gf-definition` and Fourier transform, producing the
   $(e^{-\beta K_m} + e^{-\beta K_n})$ weight.
2. Implement it as the weight-matrix-over-pole-matrix expression (`lehmann`,
   `spectral_poles`; the quadruple loop forbidden with the cost contrast stated;
   consolidation tolerance stated and the sum rule re-checked after consolidating).
3. Certify on the free four-site chain: `lehmann` from the 16-state ED against the mode
   sum $\sum_k |U_{0k}|^2/(i\omega_n - \varepsilon_k)$.
4. State the rite (prose): every many-body code in this course earns trust on a free
   problem first.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    lehmann_dev < 1e-13,
    "Lehmann certified on the free chain",
    f"max dev vs the mode sum {lehmann_dev:.1e} over n = 0, 1, 2",
)
validate.close(
    float(w_free.sum()),
    1.0,
    "and the consolidated sticks keep their sum rule",
    rtol=1e-12,
)

## Exercise 4 — The dimer's spectrum I: the free limit and the sum-rule battery

The promise of [§7.23](second-quantization.ipynb) kept, at the particle–hole point. Cite {eq}`eq-dimer-gf`.

1. Set $\mu = U/2$ and derive in two lines why it is the particle–hole point; build
   $K = H - \mu N$ from the `hubbard_dimer` of [§7.23](second-quantization.ipynb) (reused, not rebuilt).
2. Verify the free limit: $U = 0$ gives exactly two poles at $\pm t$ with weights
   $\tfrac12$ — the molecular-orbital levels re-read as addition/removal energies.
3. Run the sum-rule battery at $U = 8t$, $\beta = 4$ (`sum_rules`): $\int A = 1$, first
   moment $0$, $A(\omega) = A(-\omega)$, and $n = G(0^-) = \tfrac12$ — each with its
   one-line meaning.
4. State the battery's role (prose): four independent exact statements any spectral
   computation must pass before anything about interactions is believed.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [float(p_u0[0]), float(p_u0[1]), float(w_u0[0]), float(w_u0[1])],
    [-T_DIMER, T_DIMER, 0.5, 0.5],
    "the free limit: MO levels as addition/removal energies",
    atol=1e-10,
)
validate.close(
    [sumA, M1, n_occ],
    [1.0, 0.0, 0.5],
    "the sum-rule battery",
    atol=1e-8,
)
validate.check(
    ph_dev < 1e-12,
    "and the spectrum is even at the particle-hole point",
    f"max |A(w) - A(-w)| pole asymmetry {ph_dev:.1e}",
)

## Exercise 5 — The dimer's spectrum II: a thermal object, and the Hubbard bands
(centerpiece)

Eight poles where the textbook draws four — and why both are right. Cite
{eq}`eq-dimer-gf`.

1. Count the poles of $A(\omega)$ at $U = 8t$, $\beta = 4$ (`spectral_poles`): eight —
   and identify the satellites as transitions from the thermally populated triplet,
   the assignment made explicit from the $N = 1, 2, 3$ sector energies of $K$.
2. Teach the headline (prose plus the three-panel figure): the spectral function is a
   thermal object; textbooks draw its $T = 0$ skeleton — and the satellites are physics,
   not noise (the threshold-them-away trap named).
3. Cool to $\beta = 20$ and verify the collapse onto the four-pole skeleton at
   $\pm(\sqrt{U^2 + 16t^2} \mp 2t)/2$ with weights $w_\pm = \tfrac14(1 \pm
   4t/\sqrt{U^2+16t^2})$ (closed forms derived) — the lower and upper Hubbard bands.
4. Close the loop with [§7.23](second-quantization.ipynb) (the table): $U \leftrightarrow$ band splitting
   $\leftrightarrow$ the charge $C(T)$ peak; $J \leftrightarrow$ satellite activation
   $\leftrightarrow$ the spin peak; ARPES and STM named in one outward breath.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    n_poles_warm == 8 and satellites_present,
    "A(w) is a thermal object: eight poles at beta = 4, satellites at the triplet lines",
    f"{n_poles_warm} poles; satellites at +-3, +-5 present",
)
validate.close(
    p_skel,
    skel_exact,
    "the Hubbard bands, with their closed form",
    rtol=1e-6,
)
validate.close(
    [float(w_skel[1]), float(w_skel[0])],
    [w_exact_inner, w_exact_outer],
    "and their weights (1 +- 4t/r)/4, up to the thermal residue e^(-beta J)",
    rtol=5e-4,
)

## Exercise 6 — The FT rendezvous

Two routes to $G(i\omega_n)$ — poles and quadrature — agreeing at second order. Cite
{eq}`eq-ft-rendezvous`.

1. Compute the dimer's $G(\tau)$ at $U = 8t$, $\beta = 4$ on grids of 401, 1601, and
   6401 points (`G_tau`, ground-shifted) and Fourier transform with `ft_G` at $n = 1$.
2. Verify against the Lehmann value and confirm the second-order rate.
3. State why trapezoid suffices on $[0, \beta]$: endpoints included, and only the
   derivative jumps there.
4. Read the rendezvous (prose): the same object by residues and by quadrature — internal
   consistency demonstrated rather than assumed.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    rate_ft,
    256.0,
    "poles and quadrature agree, at the expected second-order rate",
    rtol=3e-1,
)
validate.check(
    ft_errs[2] < 1e-7,
    "and the finest grid lands below 1e-7",
    f"6401 points: {ft_errs[2]:.1e}",
)

## Exercise 7 — (STUDENT/STRETCH) The inverse problem, quantified

Two spectra that could not look more different; data that could hardly look more alike —
with the suppression law verified to three digits. Cite {eq}`eq-ill-posed`.

1. Construct the pair — weight split at $\pm0.2$ (half each) versus one peak at $0$ —
   with zeroth and first moments matched exactly (the leak trap: a mismatched $M_1$
   contributes $1/\omega_n^2$ and spoils the $\omega^{-3}$ law), and compute both
   $G(i\omega_n)$ for $n = 0\dots5$ at $\beta = 2$.
2. Tabulate $|\Delta G|$: from $10^{-2}$ at $n = 0$ to $8\times10^{-6}$ at $n = 5$.
3. Derive the moment-suppression law (expand the kernel in $1/\omega_n$: matched moments
   cancel, $\Delta G \approx \Delta M_2/\omega_n^3$) and verify it to three digits at
   every frequency but the lowest.
4. Read it precisely (prose): the kernel transmits moments and exponentially little
   else; place [§7.21](path-integral-monte-carlo.ipynb) as the benign single-pole case; name MaxEnt and Nevanlinna (Jarrell
   & Gubernatis cited; outward).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    dG_measured,
    law_values,
    "the ill-posedness as a verified law: dM2/w^3 at every frequency",
    rtol=2e-2,
)
validate.check(
    dG_measured[0] > 1e-2 * 0.9 and dG_measured[5] < 1e-5,
    "grossly different spectra, sub-1e-5 different data by n = 5",
    f"|dG|: {dG_measured[0]:.1e} -> {dG_measured[5]:.1e}",
)

## Exercise 8 — (Synthesis) Sentences

No new computation: what the alphabet said, once it could speak.

The alphabet of [§7.23](second-quantization.ipynb) made its first sentences, and they turned out to be the sentences
experiments speak. A propagator on the thermal circle, antiperiodic because fermions
anticommute, carrying a sum rule in its jump; Matsubara frequencies that spent two
notebooks as scenery and here did a day's work — including one sum whose clean
convergence to a wrong answer taught more than ten correct ones would have; a definition
made so concrete it fit in one vectorized line, certified on a free chain to sixteen
digits, and then aimed at the dimer, where the Mott gap that [§7.23](second-quantization.ipynb) read from a specific
heat reappeared as two bands in a spectrum. The same physics, before different
instruments — and the spectrum turned out to wear its temperature, growing satellite
lines the moment the triplet could be thermally reached, a fact the textbook skeleton
quietly omits.

And at the end, an honest accounting of what this object refuses to do: its
imaginary-time data forget spectra by the momentful, to three verified digits at every
frequency but the lowest. There is a maturity in learning what a tool cannot do. The Green's function
answers the experimental question exactly and forgets the spectral question
exponentially — and both facts are theorems, not moods. A computationalist who knows the
second fact to three digits will never be the one who publishes a spectrum their data
could not have contained.

One notebook remains in the gateway: perturb the system weakly, and discover that the
answer was in equilibrium's correlations all along ([§7.25](linear-response-kubo.ipynb)).

## Notebook summary

The Coda's second notebook: the propagator at temperature, made numerical claim by
claim.

- **The definition** {eq}`eq-gf-definition`: $G_{ij}(\tau) = -\langle T_\tau c_i(\tau)
  c_j^\dagger(0)\rangle$ with $K = H - \mu N$; conventions fixed once; ARPES/STM named —
  the object is measured, not merely computed.
- **The free mode** {eq}`eq-free-mode`: the closed form met at $10^{-16}$ (gated); the
  jump rule $G(0^+) + G(\beta^-) = -1$ — the anticommutator in the discontinuity —
  verified (gated); antiperiodicity derived in three lines and verified; odd-only
  Fourier content confirmed (even/odd $\sim 10^{-4}$, gated); $G(i\omega_n) =
  1/(i\omega_n - \varepsilon)$ by quadrature to six digits (gated).
- **Matsubara sums** {eq}`eq-matsubara-sums`: the naive symmetric truncation converges
  cleanly to $n_F - \tfrac12$ (gated) — the dropped $0^+$ carries exactly the jump's
  half, the sum rule returning as a failure mode; the $\pm n$ paired form converges at
  the measured $1/N$ (gated); the contour form closes three routes on one number. Kit:
  pair, know the tail, respect the $0^+$; convergence tests precision, sum rules test
  truth.
- **Lehmann** {eq}`eq-lehmann`: derived (both orderings, one pass — the $(w_m + w_n)$
  weight is the antiperiodic boundary's fingerprint); implemented as one vectorized
  weight/pole expression with the quadruple loop forbidden; poles consolidated with the
  sum rule re-checked (gated); certified on the free chain at $10^{-16}$ (gated).
- **The dimer** {eq}`eq-dimer-gf`: $\mu = U/2$ derived as the PH point; the free limit's
  two MO poles (gated); the sum-rule battery — weight 1, $M_1 = 0$, even, $n = \tfrac12$
  — all gated; at $\beta = 4$ *eight* poles with satellites assigned to the populated
  triplet (gated) — $A(\omega)$ is a thermal object; at $\beta = 20$ the four-pole
  skeleton at $\pm(\sqrt{U^2+16t^2} \mp 2t)/2$ with weights $(1 \pm 4t/r)/4$ (closed
  forms derived, gated): the Hubbard bands, the Mott gap in the spectrum, and the [§7.23](second-quantization.ipynb)
  correspondence tabled ($U \leftrightarrow$ charge peak, $J \leftrightarrow$ spin
  peak).
- **The FT rendezvous** {eq}`eq-ft-rendezvous`: trapezoid against Lehmann at second
  order, $2\times10^{-5} \to 7.7\times10^{-8}$ (gated).
- **Ill-posedness as law** {eq}`eq-ill-posed`: matched-moment spectra with
  $|\Delta G| = \Delta M_2/\omega_n^3$ verified to three digits at every frequency but the lowest
  (gated); [§7.21](path-integral-monte-carlo.ipynb) placed as the benign single-pole case; MaxEnt/Nevanlinna named.

Standing rules issued here: respect the $0^+$; pair Matsubara frequencies and estimate
the tail; re-check the sum rule after every pole consolidation; never threshold thermal
satellites away; and match moments exactly before comparing spectra.

## Outlook

- **Linear response and Kubo** ([§7.25](linear-response-kubo.ipynb)): equilibrium answering questions; response
  functions as the two-particle cousins of today's propagator.
- **Analytic continuation in practice**: MaxEnt, stochastic continuation, Nevanlinna
  (Jarrell & Gubernatis, *Phys. Rep.* **269**, 133 (1996); outward).
- **Self-energies and Dyson's equation**: perturbation theory written in the Coda's
  language (outward, named).
- **ARPES and STM**: $A(\omega)$ as laboratory data (outward, named).
- Cross-reference: [§7.2](complex-analysis-applications.ipynb) (the frequencies, put to work), [§7.20](imaginary-time-quantum-classical.ipynb) (the circle, fermionic
  edition), [§7.21](path-integral-monte-carlo.ipynb) (the benign case, placed), [§7.23](second-quantization.ipynb) (the machinery; the promise kept; the
  two-scale table), [§7.7](bose-einstein-fermi-dirac.ipynb) ($n_F$, throughout).

In [ ]:
from ecp.style import footer

footer()